In [ ]:
# === Notebook common preamble (load the llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates (when running from the notebooks folder)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Warning] load the llm_math package text: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === end preamble ===


# Ch 14. Attention text text — text·text·Value ⭐

> **Learning Goals**
> - Q, K, Vtext text text(database lookup) text text
> - Scaled Dot-Product Attentiontext $\sqrt{d_k}$text text text textdegreestext
> - Attentiontext Time/Space textdegrees $O(n^2 d)$text text
> - Attention text CPU vs GPUtext text Comparisontext

## 14.1 Attentiontext text text

RNNtext text: text text text text text text text text. text text text text.

Bahdanau Attention(2014): text text text text **text** text text text.

**Self-Attention** (text): text text text text text text text. RNN textdegrees text text Training.

## 14.2 text·text·Value (Q, K, V) — text text

Datatext text text:
- **Query (Q)**: text text text text
- **Key (K)**: text text text (text text text text text)
- **Value (V)**: text text text text

Attention: Qtext text Ktext textdegreestext Calculation → textdegreestext text V text.

text:
$$\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V$$

- $Q \in \mathbb{R}^{n \times d_k}$: $n$text text
- $K \in \mathbb{R}^{n \times d_k}$: $n$text text
- $V \in \mathbb{R}^{n \times d_v}$: $n$text Value
- $QK^\top \in \mathbb{R}^{n \times n}$: text text-text text text


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

# Scaled Dot-Product Attention text (NumPy)
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def attention(Q, K, V, mask=None):
    """Scaled Dot-Product Attention.
    Q, K, V: (n, d) or (batch, n, d)
    """
    d_k = Q.shape[-1]
    scores = Q @ np.swapaxes(K, -1, -2) / np.sqrt(d_k)  # (..., n, n)
    if mask is not None:
        scores = np.where(mask, scores, -1e9)
    weights = softmax(scores, axis=-1)
    output = weights @ V
    return output, weights

# text text
np.random.seed(0)
n, d = 4, 8
Q = np.random.randn(n, d)
K = np.random.randn(n, d)
V = np.random.randn(n, d)

output, weights = attention(Q, K, V)
print(f"Q shape: {Q.shape}")
print(f"Attention output: {output.shape}")
print(f"Attention weights: {weights.shape}")
print(f"\ntext text Sum (softmax text): {weights.sum(axis=-1).round(4)}")


## 14.3 text $\sqrt{d_k}$text text?

$QK^\top$text text $\sum_{i=1}^{d_k} Q_i K_i$. Q, Ktext $\mathcal{N}(0, 1)$text:

$$\mathrm{Var}(QK^\top_{ij}) = \sum_{i=1}^{d_k} \mathrm{Var}(Q_i K_i) = d_k$$

text text text = $\sqrt{d_k}$. $d_k$text text text text softmaxtext **text** (text Valuetext 1, text 0).

$\sqrt{d_k}$text text text 1text text → softmaxtext text Distribution text. textdegrees text text.


In [ ]:
# sqrt(d_k) text text text
np.random.seed(0)
dks = [8, 64, 512, 2048]
fig, axes = plt.subplots(1, len(dks), figsize=(16, 4))

for ax, d_k in zip(axes, dks):
    n = 4
    Q = np.random.randn(n, d_k)
    K = np.random.randn(n, d_k)
    # text text
    scores_no = Q @ K.T
    # text text
    scores_scaled = scores_no / np.sqrt(d_k)

    # softmax
    p_no = softmax(scores_no)
    p_scaled = softmax(scores_scaled)

    ax.bar(['q0', 'q1', 'q2', 'q3'], p_no[0], alpha=0.5, label='Scaling X', color='red')
    ax.bar(['q0', 'q1', 'q2', 'q3'], p_scaled[0], alpha=0.5, label='Scaling O', color='blue')
    ax.set_title(f'd_k={d_k}\nVariance: {scores_no.var():.1f} → {scores_scaled.var():.1f}')
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('../figures/ch14_scaling_effect.png', dpi=100, bbox_inches='tight')
plt.show()
print("=> d_ktext text Scaling textPlane softmaxtext text Valuetext text (text).")


## 14.4 Attentiontext Time·Space textdegrees

$$\mathrm{Attn}(Q, K, V) = \mathrm{softmax}(QK^\top / \sqrt{d_k}) V$$

- $QK^\top$: $n \times d \times n = O(n^2 d)$ text
- $\mathrm{softmax}(A) V$: $n \times n \times d = O(n^2 d)$
- **text Time**: $O(n^2 d)$
- **Space**: $A \in \mathbb{R}^{n \times n}$ → $O(n^2)$

Problem: Sequence Length $n$text text $n^2$text text. 8K text $n^2 = 6.4 \times 10^7$.
→ Flash Attention, text Attention text text (Ch 27text).


In [ ]:
# Sequence Lengthtext text Visualization
seq_lens = [128, 256, 512, 1024, 2048, 4096, 8192]
n2 = [n**2 for n in seq_lens]
n2d = [n**2 * 64 for n in seq_lens]  # d=64

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(seq_lens)), n2, alpha=0.7, label='$n^2$ (Space)')
ax.bar(range(len(seq_lens)), [n/1000 for n in n2d], alpha=0.7, label='$n^2 \\cdot d$ / 1000 (Operation)')
ax.set_xticks(range(len(seq_lens)))
ax.set_xticklabels([str(n) for n in seq_lens])
ax.set_xlabel('Sequence Length n')
ax.set_ylabel('Complexity')
ax.set_title('Attention Complexity: $O(n^2 d)$ — Sequence Lengthtext text text')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/ch14_complexity.png', dpi=100, bbox_inches='tight')
plt.show()
print("8K Contexttext 64M text Attention Scores. Flash Attentiontext text text.")


## 14.5 text text (Causal Masking)

GPT text autoregressive Modeltext text text text text text.

text Matrix $M$:
$$M_{ij} = \begin{cases} 0 & \text{if } i \geq j \text{ (text/text)} \\ -\infty & \text{if } i < j \text{ (text)} \end{cases}$$

$A + M$text softmax → text text text 0text text.


In [ ]:
# text text
n = 6
mask = np.triu(np.ones((n, n)) * -1e9, k=1)  # text text -inf
print("Causal Mask (text text -inf):")
print(mask[:4, :4])

# text text
np.random.seed(0)
scores = np.random.randn(n, n)  # Attention text
print(f"\nOriginal Scores (text text): {scores[0].round(2)}")
masked = scores + mask
print(f"Mask Application (text text): {masked[0].round(2)}")
weights = softmax(masked)
print(f"\ntext text (text text): {weights[0].round(4)}")
print(f"  → text text text text text text (1.0)")

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
im0 = axes[0].imshow(scores, cmap='viridis'); plt.colorbar(im0, ax=axes[0])
axes[0].set_title('Original Attention Scores')
im1 = axes[1].imshow(mask != -1e9, cmap='binary'); plt.colorbar(im1, ax=axes[1])
axes[1].set_title('Causal Mask (text=text)')
im2 = axes[2].imshow(weights, cmap='Blues'); plt.colorbar(im2, ax=axes[2])
axes[2].set_title('Mask Application text Weight')
plt.tight_layout()
plt.savefig('../figures/ch14_causal_mask.png', dpi=100, bbox_inches='tight')
plt.show()


## 14.6 [CPU/GPU Benchmark ⑤] Sequence Lengthtext Attention Time

Sequence Length $n$text 2text text Attention Timetext 4text (text 4text).
CPU vs GPU text text text text.


In [ ]:
# Attention Benchmark
import time
from llm_math.bench import time_fn, format_results_table

def attention_torch(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-1, -2) / (d_k ** 0.5)
    if mask is not None:
        scores = scores + mask
    weights = F.softmax(scores, dim=-1)
    return weights @ V

# Sequence Lengthtext
print(f"{'n':>6} | {'CPU (ms)':>12} | {'GPU (ms)':>12} | {'Speedup':>10}")
print("-" * 50)
for n in [128, 256, 512, 1024, 2048]:
    d = 64
    Q = torch.randn(n, d)
    K = torch.randn(n, d)
    V = torch.randn(n, d)
    res_cpu = time_fn(attention_torch, Q, K, V, device='cpu', warmup=2, repeat=3)
    if torch.cuda.is_available():
        Q_g, K_g, V_g = Q.cuda(), K.cuda(), V.cuda()
        res_gpu = time_fn(attention_torch, Q_g, K_g, V_g, device='cuda', warmup=3, repeat=5)
        speedup = res_cpu['mean_ms'] / res_gpu['mean_ms']
        print(f"{n:>6} | {res_cpu['mean_ms']:>12.3f} | {res_gpu['mean_ms']:>12.3f} | {speedup:>9.2f}x")
    else:
        print(f"{n:>6} | {res_cpu['mean_ms']:>12.3f} | {'N/A':>12} | {'N/A':>10}")

if not torch.cuda.is_available():
    print("\n⚠ GPU text. n=2048 textdegreesPlane CPUtext text text text.")


In [ ]:
# PyTorch SDPA (scaled_dot_product_attention) text
print("PyTorch SDPA Function text:")
print("(text Flash Attention text Optimization text text Linetext)\n")

n, d = 512, 64
Q = torch.randn(1, n, d)
K = torch.randn(1, n, d)
V = torch.randn(1, n, d)

# text text
out_manual = attention_torch(Q, K, V)

# PyTorch SDPA (F.scaled_dot_product_attention)
out_sdpa = F.scaled_dot_product_attention(Q, K, V)

print(f"text Implementation vs SDPA Maximum Error: {(out_manual - out_sdpa).abs().max():.2e}")
print("\n=> text Result. SDPAtext text Optimizationtext text.")

# Speed Comparison
print("\nSpeed Comparison (n=512):")
t_manual = time_fn(attention_torch, Q, K, V, device='cpu', warmup=2, repeat=5)
def sdpa_call(Q, K, V):
    return F.scaled_dot_product_attention(Q, K, V)
t_sdpa = time_fn(sdpa_call, Q, K, V, device='cpu', warmup=2, repeat=5)
print(f"  text Implementation: {t_manual['mean_ms']:.3f} ms")
print(f"  PyTorch SDPA: {t_sdpa['mean_ms']:.3f} ms")
print(f"  Speed Improvement: {t_manual['mean_ms'] / t_sdpa['mean_ms']:.2f}x")


## 14.7 Attention Backpropagation (text)

text text text text text. text:

$$\frac{\partial \mathcal{L}}{\partial Q} = \frac{1}{\sqrt{d_k}} \left(\frac{\partial \mathcal{L}}{\partial A} V K^\top + \ldots\right)$$

PyTorchtext `autograd`text text text. Flash Attentiontext text Backpropagationtext text text (Ch 27).

## 14.8 Key Takeaways

| text | text | text |
|---|---|---|
| Attention | $\mathrm{softmax}(QK^\top/\sqrt{d_k})V$ | text-text textdegreestext Value text |
| text | $\frac{1}{\sqrt{d_k}}$ | text 1text text |
| text text | $M_{ij} = -\infty$ if $i < j$ | text text text |
| textdegrees | $O(n^2 d)$ Time, $O(n^2)$ Space | Sequence Lengthtext 2text |

## Exercises

1. $\mathrm{Var}(QK^\top_{ij}) = d_k$text Q, Ktext text iid $\mathcal{N}(0,1)$text text textdegreestext.
2. text text text Attentiontext text Attentiontext Outputtext Comparisontext.
3. Attention text Matrixtext Visualizationtext, text text text text text.
4. Sequence Length 256, 512, 1024, 2048text CPU Timetext text $O(n^2)$text Verificationtext.
5. PyTorch SDPAtext `is_causal=True` text text text Attentiontext text.

> Solutions: `solutions/ch14_solutions.ipynb`
